# A/B-тест страницы деталей товара

Проанализируйте результаты A/B-теста. Рассчитайте метрики и проверьте статистическую значимость их изменений:

- ARPPU;
- конверсия в покупку;
- ARPU;
- конверсия в добавление в корзину;
- среднее количество заказов на пользователя.

Подготовьте результаты и рекомендацию для продуктовой команды. Используйте двусторонние гипотезы и уровень значимости 0,05.

## Подготовка
**Jupyter:** распакуйте архив и откройте ноутбук. Оба CSV должны лежать рядом.

**Colab:** загрузите ноутбук, затем `participants.csv` и `events.csv` через панель «Файлы».

Если библиотек нет, раскомментируйте следующую ячейку. Установка требует интернета.

In [ ]:
# %pip install "numpy>=1.26" "pandas>=2.1" "scipy>=1.11" "statsmodels>=0.14" "matplotlib>=3.8"

`pandas` используем для таблиц, `numpy` — для вычислений, `scipy.stats` — для распределений и тестов.
Новые статистические функции разберём перед использованием.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt


participants = pd.read_csv('participants.csv')
events = pd.read_csv('events.csv', parse_dates=['event_time'])

In [ ]:
participants.head()

In [ ]:
events.head()

## 1. ARPPU
С помощью t-теста проверьте, статистически значимо ли изменение ARPPU.

**ARPPU — суммарная выручка за период на одного покупателя.** Один покупатель — одно наблюдение, даже если у него несколько заказов. Непокупатели в эту метрику не входят.

### 1.1. Подготовьте данные
Оставьте только покупки, рассчитайте выручку каждого покупателя и присоедините его группу A/B.

<details><summary>Подсказка по pandas</summary>

Выберите `event_type == 'purchase'`, затем используйте `groupby('user_id')` и `reset_index()`.
Группу присоедините через `merge()`.

</details>

In [ ]:
# Оставьте только события покупки.
purchases = ...

# Получите таблицу: одна строка — один покупатель.
# Столбцы: user_id и revenue — суммарная выручка этого покупателя за весь период. Используйте reset_index() в конце
pay_revenue = ...

# Теперь самостоятельно присоедините A/B-группы из participants с помощью merge().
# В pay_revenue должны остаться столбцы user_id, revenue и group.
pay_revenue = ...

### 1.2. Разделите данные на группы
Разделите пользователей на две группы в зависимости от их принадлежности к контролю и тесту. Получите `a_pay` и `b_pay` — значения выручки покупателей групп A и B.

<details><summary>Подсказка</summary>

Отфильтруйте строки по `group` и выберите столбец `revenue`.

</details>

In [ ]:
a_pay = ...
b_pay = ...

### 1.3. Посчитайте средние
Для каждой группы найдите число покупателей $n$ и среднюю выручку $\bar{x}$.
Рассчитайте разность $\widehat\Delta=\bar{x}_B-\bar{x}_A$.

<details><summary>Подсказка</summary>

Используйте `len()` и `.mean()`.

</details>

In [ ]:
n_a, n_b = ..., ...
mean_a, mean_b = ..., ...
delta_pay = ...
mean_a, mean_b, delta_pay

### 1.4. Посчитайте стандартное отклонение
Стандартное отклонение описывает разброс выручки покупателей вокруг среднего и измеряется в рублях:
$$s=\sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}}.$$

Используйте `.std(ddof=1)`: параметр `ddof=1` задаёт деление на $n-1$ под корнем.
Сохраните результаты для A и B в `s_a` и `s_b`. Их квадраты — дисперсии, которые понадобятся в следующем шаге.

In [ ]:
s_a = ...
s_b = ...

print(f"Стандартное отклонение: в группе А {s_a}, в группе B {s_b}")

### 1.5. Посчитайте стандартную ошибку
Используем обычный двухвыборочный **t-тест Стьюдента**. В этом упражнении принимаем равенство дисперсий в группах как допущение. Пользователи независимы

Сначала объединим оценки дисперсий:
$$s_p^2=\frac{(n_A-1)s_A^2+(n_B-1)s_B^2}{n_A+n_B-2}.$$
Здесь $s_A^2$ и $s_B^2$ — квадраты стандартных отклонений: `s_a**2` и `s_b**2`. Сохраните общую оценку в `pooled_var`.

Затем рассчитайте стандартную ошибку разности средних:
$$SE=\sqrt{s_p^2\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}.$$
$SE$ характеризует неопределённость разности средних. 
<details><summary>Подсказка</summary>

Сначала вычислите `pooled_var`, затем `se_pay`. Для квадратного корня используйте `np.sqrt()`.

</details>

In [ ]:
pooled_var = ...
se_pay = ...
se_pay

### 1.6. Посчитайте t-статистику
Сформулируйте H₀ и H₁ для разности средних ARPPU. При H₀ об отсутствии изменения:
$$t=\frac{\widehat\Delta-0}{SE}.$$

Статистика показывает, на сколько стандартных ошибок наблюдаемая разность отстоит от нуля.

In [ ]:
t_stat = ...
t_stat

### 1.7. Степени свободы
Для двухвыборочного t-теста Стьюдента:
$$df=n_A+n_B-2.$$


In [ ]:
df_pay = ...
df_pay

### 1.8. Найдите p-value в Python
`stats.t.sf(x, df)` возвращает $P(T>x)$ — площадь справа от x под t-распределением с `df` степенями свободы.

Для двусторонней проверки учитываем оба хвоста:
$$p\text{-value}=2\,\mathrm{sf}(|t_{obs}|;df).$$


**Рассчитайте p-value с помощью `stats.t.sf`, используя `t_stat` и `df_pay`. Сделайте вывод при уровне значимости 0,05.**

p-value — вероятность получить такую же или более экстремальную статистику при верной H₀ и предпосылках теста.

In [ ]:
p_pay = ...  # используйте t_stat, df_pay, abs и stats.t.sf
print(f'p-value = {p_pay:.6f}')

## 2. Конверсия в покупку
С помощью z-теста двух долей проверьте, статистически значимо ли изменение конверсии.

### 2.1. Посчитайте покупателей и участников
Для каждой группы найдите число уникальных покупателей $x$ и общее число участников $n$.
Числа покупателей уже рассчитаны в разделе 1 — используйте их. Один пользователь с несколькими заказами считается одним покупателем. В знаменателе — все участники группы.

<details><summary>Подсказка</summary>

Группы возьмите из `participants`, покупки — из `events`. Можно объединить таблицы через `merge()` или отобрать ID через `.isin()`.
Уникальных покупателей считайте через `.nunique()`.

</details>

Для нормального приближения нужны независимые пользователи и достаточно исходов: ориентир — не меньше **10 покупателей и 10 непокупателей в каждой группе**.

In [ ]:
x_a = ...       # число покупателей в A
x_b = ...       # число покупателей в B
n_a_conv = ...  # число участников A
n_b_conv = ...  # число участников B

### 2.2. Посчитайте конверсии и изменение
$$\hat p_A=\frac{x_A}{n_A},\qquad \hat p_B=\frac{x_B}{n_B}.$$
Найдите разность B − A в процентных пунктах и относительное изменение в процентах.
Например, 20% → 19% — это −1 п.п. и −5% относительно контроля.

<details><summary>Подсказка</summary>

Разность долей умножьте на 100. Для относительного изменения разделите разность на долю в A и умножьте на 100.

</details>

In [ ]:
p_a = ...
p_b = ...
# Абсолютное изменение конверсии B − A в процентных пунктах.
delta_pp = ...
# Относительное изменение конверсии к группе A, в процентах.
lift_conv_pct = ...

### 2.3. Рассчитайте объединённую долю
Сформулируйте H₀ и H₁. При H₀ доля покупателей в группах одинакова. Оценим её по объединённым данным:
$$\hat p_{pool}=\frac{x_A+x_B}{n_A+n_B}.$$

In [ ]:
p_pool = ...

### 2.4. Рассчитайте стандартную ошибку
Используйте объединённую долю:
$$SE_0=\sqrt{\hat p_{pool}(1-\hat p_{pool})\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}.$$

<details><summary>Подсказка</summary>

Для квадратного корня используйте `np.sqrt()`.

</details>

In [ ]:
se_conv = ...

### 2.5. Рассчитайте z-статистику
$$z=\frac{\hat p_B-\hat p_A}{SE_0}.$$
В числителе используйте доли, а не проценты.

In [ ]:
z_stat = ...

### 2.6. Найдите p-value
При H₀ z-статистика приближённо следует стандартному нормальному распределению.
`stats.norm.sf(x)` возвращает $P(Z>x)$ для этого распределения. Для двух хвостов:
$$p\text{-value}=2\,\mathrm{sf}(|z|).$$
Рассчитайте p-value и сделайте вывод при уровне значимости 0,05.

In [ ]:
p_conv = ...

**Вывод:** укажите конверсию каждой группы, изменение и результат проверки гипотезы.

Ваш вывод: …

## 3. ARPU
ARPU — средняя выручка за период на всех участников эксперимента. Проверьте её изменение готовым t-тестом.

### 3.1. Добавьте непокупателей
В разделе 1 вы уже рассчитали выручку покупателей в `pay_revenue`. Используйте эту таблицу: присоедините выручку к полному списку участников и заполните нулями значения для непокупателей.

<details><summary>Подсказка</summary>

Из `pay_revenue` понадобятся `user_id` и `revenue`. Группа уже есть в `participants`.
Выберите тип `merge()`, который сохранит всех участников.
При полной выгрузке отсутствие покупок означает нулевую выручку; неизвестная сумма существующего заказа — нет.

</details>

In [ ]:
# Присоедините выручку из pay_revenue к participants.
# Сохраните всех участников; группа уже есть в participants.
# Ожидаемые столбцы: user_id, group, revenue.
arpu_data = ...
# У непокупателей после объединения выручка не заполнена.
# Замените эти пропуски нулями: в полной выгрузке отсутствие покупок означает 0.
arpu_data['revenue'] = ...

### 3.2. Разделите данные на группы
Получите выручку участников A и B. В обеих выборках должны остаться нули непокупателей.

In [ ]:
a_arpu = ...
b_arpu = ...

### 3.3. Посчитайте ARPU и изменение
Найдите среднее в каждой группе, разность B − A в рублях и относительное изменение в процентах.

In [ ]:
arpu_a = ...
arpu_b = ...
delta_arpu = ...
lift_arpu_pct = ...

### 3.4. Проверьте гипотезу готовым t-тестом
`stats.ttest_ind()` получает две выборки и возвращает результат теста:

- первые два аргумента — значения метрики по пользователям; передайте сначала B, затем A;
- `equal_var=True` — обычный t-тест Стьюдента с предположением равенства дисперсий;
- `alternative='two-sided'` — двусторонняя проверка.

У результата `.statistic` содержит t-статистику, `.pvalue` — p-value, `.df` — степени свободы.

Вызовите функцию для ARPU. Сохраните результат в **`arpu_result`**: он понадобится для доверительного интервала. Выведите статистику и p-value.

<details><summary>Как вывести части результата</summary>

Обращайтесь к полям через точку:
```python
print(arpu_result.statistic)  # t-статистика
print(arpu_result.pvalue)     # p-value
print(arpu_result.df)         # степени свободы
```
Значения можно сохранить в переменные, например `t_arpu = arpu_result.statistic`.

</details>

In [ ]:
arpu_result = ...
t_arpu = ...  # извлеките поле statistic
p_arpu = ...  # извлеките поле pvalue

# Выведите полученные значения с помощью print().

## 4. Доверительный интервал ARPU
Оценка разности средних зависит от случайного состава выборки. **Доверительный интервал (ДИ)** показывает её неопределённость: какие значения истинной разности согласуются с данными при предпосылках метода.

**Что означает 95%?** Если многократно повторять эксперимент и строить интервал одним способом, примерно 95% полученных интервалов накроют истинную разность. Параметр фиксирован, а границы меняются от выборки к выборке.
Это не вероятность попадания параметра в уже рассчитанный интервал и не диапазон выручки 95% пользователей.

Для t-теста интервал имеет вид:
$$\text{разность средних}\ \pm\ t_{0.975,df}\times SE.$$
Критическое значение оставляет по 2,5% в каждом хвосте. При прочих равных больше наблюдений сужает интервал, больший разброс или уровень доверия расширяет его.

### 4.1. Получите интервал готовой функцией
У результата t-теста есть метод **`.confidence_interval(confidence_level=0.95)`**.
Он возвращает границы `.low` и `.high` разности средних. Поскольку в тест передали B, затем A, интервал относится к B − A, в рублях на участника.

Код готов — выполните ячейку.

In [ ]:
arpu_ci = arpu_result.confidence_interval(confidence_level=0.95)
print(f'95% ДИ разности ARPU: [{arpu_ci.low:.2f}; {arpu_ci.high:.2f}] руб.')

### 4.2. Посмотрите на интервал
Точка — наблюдаемая разность, отрезок — 95%-й ДИ, пунктир — ноль. Код графика дан готовым.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.errorbar(b_arpu.mean()-a_arpu.mean(), 0,
            xerr=[[(b_arpu.mean()-a_arpu.mean())-arpu_ci.low],
                  [arpu_ci.high-(b_arpu.mean()-a_arpu.mean())]],
            fmt='o', color='#7545db', capsize=7, linewidth=3)
ax.axvline(0, color='#555555', linestyle='--')
ax.set_yticks([])
ax.set_xlabel('Разность ARPU B − A, руб. на участника за 14 дней')
ax.set_title('Оценка и 95%-й доверительный интервал')
plt.tight_layout()
plt.show()

### 4.3. Интерпретируйте результат
1. Входит ли ноль в интервал? Как это согласуется с p-value?
2. Какие размеры снижения и роста допускает интервал?
3. Можно ли утверждать, что вариант B выгоднее? А что эффект отсутствует?

Для соответствующего двустороннего t-теста при α = 0,05 ноль вне 95%-го ДИ означает p-value < 0,05.
Если ноль внутри, H₀ не отвергаем. Это не доказывает отсутствие эффекта.

Ваш ответ: …

## 5. Конверсия в добавление в корзину
Рассчитайте конверсию в добавление в корзину для A и B, абсолютное и относительное изменения. Проверьте статистическую значимость с помощью z-теста двух долей и сформулируйте вывод.

**Справка по новой функции:** `proportions_ztest` из `statsmodels.stats.proportion` принимает `count` — числа пользователей с успехом, `nobs` — размеры групп в том же порядке. `value=0` задаёт разность при H₀, `alternative='two-sided'` — двустороннюю проверку. Возвращает z-статистику и p-value.

In [ ]:
# Ваше решение

## 6. Среднее количество заказов на пользователя
Рассчитайте среднее количество заказов на пользователя для A и B, абсолютное и относительное изменения. Проверьте статистическую значимость с помощью обычного двухвыборочного t-теста Стьюдента и сформулируйте вывод.

Используйте таблицу покупок `purchases`, подготовленную в разделе 1.

In [ ]:
# Ваше решение

## 7. Результаты для команды
### 7.1. Заполните таблицу
В шаблоне ниже замените `...` уже рассчитанными значениями. Для конверсий укажите A и B в процентах, абсолютное изменение — в процентных пунктах. Для денежных метрик — в рублях.
Относительное изменение всех метрик укажите в процентах. Доверительный интервал нужен только для ARPU.

In [ ]:
# Вместо ... вставьте значения или переменные из предыдущих заданий.
results = pd.DataFrame([
    ['ARPPU, руб.', ..., ..., ..., ..., ..., '—'],
    ['Конверсия в покупку, %; Δ в п.п.', ..., ..., ..., ..., ..., '—'],
    ['ARPU, руб.', ..., ..., ..., ..., ..., ...],
    ['Конверсия в корзину, %; Δ в п.п.', ..., ..., ..., ..., ..., '—'],
    ['Заказы на пользователя', ..., ..., ..., ..., ..., '—']
], columns=[
    'Метрика', 'A', 'B', 'B − A', 'Изменение, %', 'p-value', '95% ДИ разности'
])
results

### 7.2. Сформулируйте рекомендацию
Напишите 4–6 предложений: что изменилось, насколько определён результат по ARPU и стоит ли запускать B.
Чего не хватает для решения? Что предложите сделать дальше?

Основная метрика — ARPU; остальные помогают понять результат. Несколько номинальных p-value не контролируют общую вероятность ложного вывода.
Если данных недостаточно, нельзя просто продолжать сбор до p < 0,05. Для нового эксперимента заранее определяют полезный эффект, размер выборки и правило решения.

**Наблюдаемые изменения.** ARPPU …; конверсия в покупку …; ARPU … . Корзина и число заказов … .

**Статистический вывод.** Для … H₀ отвергаем; для … не отвергаем при α = 0,05. Доверительный интервал разности ARPU: … . Это означает … .

**Рекомендация.** Предлагаем …, потому что … .

**Что нужно дальше.** Для решения не хватает … . Следующий шаг — … .

### Вопросы для обсуждения
- Почему среднее по строкам покупок не даёт ARPPU?
- Что получится, если при расчёте ARPU использовать `inner join`?
- Почему z-тесту недостаточно передать только две конверсии без размеров групп?
- Что изменится в интервале при уровне доверия 99%?
- Может ли дополнительный покупатель с маленькой суммой снизить ARPPU и одновременно увеличить ARPU?
- Открытия карточек есть в данных, но не входят в обязательный анализ. Почему повторные открытия после знакомства с вариантом уже могут зависеть от воздействия?

### Справка
- [SciPy: ttest_ind](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html)
- [SciPy: t-распределение, sf и ppf](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html)
- [SciPy: нормальное распределение](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [statsmodels: proportions_ztest](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html)
- [NIST: интерпретация доверительных интервалов](https://www.itl.nist.gov/div898/handbook/eda/section3/eda352.htm)